<a href="https://colab.research.google.com/github/cris959/automatizando-analisis-datos-agentes/blob/clase-01/agentes_con_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Agregamos encoding y especificamos el separador por si las dudas
df = pd.read_csv('/content/datos_entregas.csv', sep=',', encoding='utf-8')

# Mostramos los primeros 5 registros
df.head()

In [ ]:
from google.colab import userdata
import os

# Recuperamos la API Key desde los secretos de Colab
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

if GROQ_API_KEY:
    # La seteamos como variable de entorno para que LangChain/Groq la detecten automáticamente
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("✅ GROQ_API_KEY cargada correctamente y seteada en el entorno.")
else:
    print("❌ Error: No se pudo cargar GROQ_API_KEY.")
    print("Verificá si agregaste la clave en el panel de llaves (Secrets) a la izquierda y activaste el acceso.")

Como estamos usando **LangChain 0.3.24**, para evitar cualquier problema de compatibilidad con las versiones cruzadas de los paquetes, es una buena práctica forzar la instalación de la biblioteca de Groq para que se alinee con la versión principal del ecosistema.

In [ ]:
!pip install langchain-groq>=0.2.0 -q

El nombre del modelo está desactualizado: El modelo **llama3-70b-8192** fue discontinuado.

In [ ]:
from langchain_groq import ChatGroq

# Instanciamos el modelo actualizado de la familia Llama 3
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
# Definimos el prompt de forma clara y directa
prompt = (
    "Tengo un dataframe llamado 'df' con las columnas 'años_experiencia_colaborador' y 'tiempo_entrega'. "
    "Escribe el código Python con la biblioteca pandas para calcular la correlación entre las dos columnas. "
    "Devuelve únicamente el bloque de código en formato Markdown, sin explicaciones."
)

# Invocamos al modelo
ai_msg = llm.invoke(prompt)

# Imprimimos la respuesta en una línea nueva
print(ai_msg.content)

In [ ]:
# 1. Calculamos la correlación
correlacion = df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])

# 2. Imprimimos el resultado en una nueva línea, limitando a 2 decimales
print(f"La correlación entre 'años_experiencia_colaborador' y 'tiempo_entrega' es: {correlacion:.2f}")

Como estamos trabajando sobre la base de **LangChain 0.3.x**, es fundamental asegurar que langchain-experimental (que contiene el agente de Pandas que vas a usar a continuación) no te instale versiones viejas de los componentes del núcleo por error.

Es un conflicto menor: La diferencia entre **2.32.4** y **2.34.2** en requests es mínima y no va a romper ninguna funcionalidad visual de tu cuaderno.

In [ ]:
!pip install langchain-experimental>=0.3.0 -q

Para que la pantalla no te quede sucia con ese bloque de texto rojo gigante cada vez que ejecutes la celda, podés silenciar los avisos de Python agregando estas dos líneas arriba de todo en tu código

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from langchain_experimental.tools import PythonAstREPLTool

# 1. Instanciamos la herramienta pasándole tu dataframe 'df'
herramienta_py = PythonAstREPLTool(locals={"df": df})

# 2. Ejecutamos el comando y capturamos la salida
resultado = herramienta_py.invoke("df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])")

# 3. Mostramos el resultado en consola
print("Resultado de la herramienta:", resultado)

In [ ]:
from langchain_core.tools import tool
from langchain_experimental.tools import PythonAstREPLTool

# 1. Creamos la herramienta REPL clásica
repl_tool = PythonAstREPLTool(locals={"df": df})

# 2. La envolvemos en una función compatible con la API de herramientas de LangChain
@tool
def ejecutar_codigo_python(codigo: str) -> str:
    """
    Ejecuta código de Python y Pandas en un entorno aislado y devuelve el resultado de la consola.
    Úsala para hacer cálculos numéricos, correlaciones, agrupar datos o generar análisis estadísticos sobre el dataframe 'df'.
    """
    return repl_tool.invoke(codigo)

# 3. Vinculamos la nueva herramienta compatible al LLM forzando su uso
llm_con_herramienta = llm.bind_tools([ejecutar_codigo_python], tool_choice="ejecutar_codigo_python")

# 4. Definimos el prompt limpio
prompt = (
    "Tengo un dataframe llamado 'df' con las columnas 'años_experiencia_colaborador' y 'tiempo_entrega'. "
    "Calcula la correlación entre las dos columnas utilizando la herramienta de Python."
)

# 5. Invocamos al modelo
respuesta = llm_con_herramienta.invoke(prompt)

# 6. Inspeccionamos las llamadas a herramientas generadas por el modelo
respuesta.tool_calls

In [ ]:
# 1. Extraemos el string del código exacto que generó el modelo
codigo_sugerido = respuesta.tool_calls[0]['args']['codigo']

print("--- Código enviado a la herramienta ---")
print(codigo_sugerido)
print("---------------------------------------\n")

# 2. Invocamos directamente a la herramienta REPL original (repl_tool)
# Pasamos el string completo. La herramienta se encarga de procesar las múltiples líneas.
resultado_final = repl_tool.invoke(codigo_sugerido)

print("El agente ejecutó el código y obtuvo:")
print(resultado_final)

In [ ]:
from langchain_core.output_parsers.openai_tools import JsonOutputKeyToolsParser

# 1. Configuramos el parser apuntando al nombre exacto de nuestra función/herramienta
parser = JsonOutputKeyToolsParser(key_name="ejecutar_codigo_python", first_tool_only=True)

# 2. Armamos la cadena usando LCEL (LangChain Expression Language)
cadena = llm_con_herramienta | parser

# 3. Invocamos la cadena con el prompt
respuesta = cadena.invoke("""
              Tengo un dataframe llamado 'df' con las columnas 'años_experiencia_colaborador' y 'tiempo_entrega'.
              Escribe el código Python con la biblioteca pandas para calcular la correlación entre las dos columnas.
              Devuelve el script de Python únicamente.
              """)

# 4. Accedemos a la clave correcta que generó el modelo ('codigo')
print("Código extraído automáticamente por la cadena:")
print(respuesta["codigo"])

In [ ]:
# Calculamos el coeficiente de correlación de Pearson
corr_coef = df['años_experiencia_colaborador'].corr(df['tiempo_entrega'])

# Mostramos el resultado en la consola
print(corr_coef)

In [ ]:
df.columns.to_list()

**inspect.cleandoc**: Es un truco genial en Python. Remueve la identación del bloque de texto pero te permite mantener el código ordenado visualmente en tu celda de Colab. El modelo recibirá el texto pegado al margen izquierdo de forma impecable.

Eliminar el **;**: Al permitirle saltos de línea normales, el modelo generará un script estándar de Python. Cuando este script pase por tu cadena con el **JsonOutputKeyToolsParser** y llegue a la herramienta **repl_tool.invoke()**, se ejecutará a la perfección

In [ ]:
import inspect
from langchain_core.prompts import ChatPromptTemplate

# 1. Definimos el prompt del sistema limpio y sin identación fantasma
system_content = inspect.cleandoc(f"""
    Tienes acceso a un dataframe de pandas llamado `df`.
    Aquí está la lista de sus columnas: {df.columns.to_list()}

    Dada una pregunta del usuario, tu única tarea es escribir el código Python necesario para responderla.

    Reglas estrictas:
    - Utiliza únicamente las bibliotecas incorporadas de Python y pandas.
    - Devuelve ÚNICAMENTE el código Python ejecutable, sin explicaciones, sin bloques de código Markdown (```), ni texto adicional.
""")

# 2. Creamos la plantilla del prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", system_content),
    ("human", "{question}")
])

In [ ]:
from langchain_core.runnables import RunnableLambda

# 1. Creamos el parser apuntando a nuestra herramienta ejecutable
parser = JsonOutputKeyToolsParser(key_name="ejecutar_codigo_python", first_tool_only=True)

# 2. Envolvemos la herramienta física en un Runnable para que acepte el operador '|'
# Extraemos la clave 'codigo' que sale del parser y se la mandamos a la herramienta
ejecutor = RunnableLambda(lambda inputs: repl_tool.invoke(inputs["codigo"]))

# 3. Armamos la súper cadena automatizada
cadena_completa = prompt | llm_con_herramienta | parser | ejecutor

# 4. Invocamos la cadena pasando la pregunta en el diccionario
respuesta = cadena_completa.invoke({
    "question": "¿Cuál es la correlación entre años experiencia del colaborador y tiempo de entrega?"
})

# 5. Imprimimos el resultado final directo en una nueva línea
print("El Agente automatizado calculó el siguiente resultado:")
print(respuesta)

In [ ]:
# 1. Invocamos la cadena completa pasando la nueva pregunta sobre el clima
respuesta = cadena_completa.invoke({
    "question": "Calcula el promedio de tiempo de entrega para cada clima?"
})

# 2. Imprimimos el resultado en una línea nueva
print("Resultado del análisis por clima:")
print(respuesta)